# 01 – Text-to-Speech (TTS) Benchmark: Edge-TTS, PiperTTS & Tacotron 2

In diesem Notebook werden drei Text-to-Speech-Systeme anhand identischer deutscher und englischer Referenztexte evaluiert:

1. **Edge-TTS (`edge-tts`)** (Cloud-basierte Microsoft Neural Voices: `de-DE-KatjaNeural` & `en-US-AriaNeural`)
2. **PiperTTS** (Leichtgewichtiges, lokales/embedded-fähiges neuronales TTS-System)
3. **Tacotron 2 + WaveGlow** (NVIDIA TorchHub Seq2Seq-Mel-Spektrogramm-Generator + WaveGlow-Vocoder)

**Bewertungskriterien (jeweils 0–5 Punkte):**
- **Sprachqualität:** Deutlichkeit (*Clarity*), Natürlichkeit (*Naturalness*), Flüssigkeit (*Smoothness*)
- **Sprachrhythmus & linguistische Intelligenz:** Interpunktion, Abkürzungen & Symbole (`Dr.`, `%`, `€`/`$`), Zahlen- & Datumsformate
- **Anpassbarkeit:** Stimmenauswahl, Sprechgeschwindigkeit, Tonhöhe & Emotionssteuerung

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ref_path = Path("../data/reference_texts.json")
with ref_path.open("r", encoding="utf-8") as f:
    ref_data = json.load(f)

print("Referenzdaten geladen für Sprachen:", list(ref_data["texts"].keys()))

---
## 1. Edge-TTS

**Setup & Besonderheiten:**
- Nutzt die Python-Bibliothek `edge-tts` (in Kombination mit `nest_asyncio` für asynchrone Ausführung in Jupyter Notebooks), die direkt auf die Microsoft Speech API zugreift.
- Bietet moderne neuronale Stimmen für zahlreiche Sprachen (u. a. `de-DE-KatjaNeural` für Deutsch und `en-US-AriaNeural` für Englisch) mit anpassbarem Sprechtempo (`rate`), Tonhöhe (`pitch`) und Lautstärke (`volume`).
- Benötigt eine aktive Internetverbindung, kommt dafür aber ohne lokale GPU oder manuelle Modell-Downloads aus.

### Referenz-Implementierung (Edge-TTS)
```python
import asyncio
import edge_tts

async def synthesize_edge(text: str, voice: str, output_path: str) -> None:
    communicate = edge_tts.Communicate(text, voice=voice)
    await communicate.save(output_path)

await synthesize_edge(german_text, "de-DE-KatjaNeural", "audiofiles/edge_german.wav")
await synthesize_edge(english_text, "en-US-AriaNeural", "audiofiles/edge_english.wav")
```

### Evaluierung & Bewertung

- **Englisch (5.0 / 5 – Einzelwertung 8.75 / 10):**  
  Sehr klare, präzise Artikulation (`en-US-AriaNeural`), subtile Intonation sowie natürliche Pausen an Kommas und Punkten. Fragen heben die Tonhöhe stimmig an; Zahlen (`1,278`, `92.4 percent`, `950$`), Abkürzungen (`Dr.`, `a.m.`) und zusammengesetzte Begriffe werden fehlerfrei ausgesprochen.
- **Deutsch (4.0 / 5 – Einzelwertung 8.0 / 10):**  
  Hohe Deutlichkeit (`de-DE-KatjaNeural`), gleichmäßiger Sprachfluss und zuverlässige Umsetzung von Interpunktion, Zahlen und Abkürzungen. Minimal synthetischer Unterton sowie leichte Abgehacktheit bei langen Komposita (z. B. *„Sprachqualitätstest“*). Keine explizite Emotionssteuerung verfügbar.

---
## 2. PiperTTS

**Setup & Besonderheiten:**
- Leichtgewichtiges, schnelles neuronales TTS-System (ONNX-basiert), das sowohl lokal (z. B. auf eingebetteten Geräten wie einem Raspberry Pi) als auch webbasiert (`https://piper.ttstool.com/`) genutzt werden kann.
- Bietet eine große Auswahl an Stimmen je Sprache sowie Tonhöheneinstellungen, erlaubt in der getesteten Standardkonfiguration jedoch keine Anpassung der Sprechgeschwindigkeit oder Emotionssteuerung.

### Referenz-Implementierung (PiperTTS)
```python
import subprocess
from IPython.display import Audio

# Lokale Synthese über die Piper-CLI (ONNX-Stimmmodell)
subprocess.run(
    ["piper", "--model", "de_DE-thorsten-medium.onnx", "--output_file", "audiofiles/Pipertts_german.wav"],
    input=german_text.encode("utf-8"),
    check=True,
)

Audio(filename="audiofiles/Pipertts_german.wav")
```

### Evaluierung & Bewertung

- **Englisch (4.0 / 5):**  
  Klare und flüssige Sprachausgabe ohne verschluckte Silben oder hörbare Artefakte. Interpunktion, Abkürzungen (`Dr.`), Zahlen (`2024`, `92.4 percent`), Uhrzeiten (`8:45 a.m.`) und Währungssymbole (`950$`) werden sprachtypisch korrekt gelesen. Wirkt insgesamt technisch-neutral bzw. monoton ohne emotionale Tiefe.
- **Deutsch (3.0 / 5):**  
  Grundsätzlich verständliche Artikulation und flüssige Wiedergabe mit korrekter Erkennung von `%`- und `€`-Symbolen, jedoch deutliche Schwächen bei der linguistischen Verarbeitung: **„Dr.“ wird buchstabenweise („D-r“) statt „Doktor“** ausgesprochen, Datumsangaben (**„12. März 2024“**) klingen unnatürlich, und die Stimme wirkt mechanisch-monoton.

---
## 3. Tacotron 2 + WaveGlow

**Setup & Besonderheiten:**
- Zweistufige Deep-Learning-Pipeline über `torch.hub` (`NVIDIA/DeepLearningExamples:torchhub`): **Tacotron 2** (Seq2Seq mit Attention) erzeugt ein Mel-Spektrogramm, das vom **WaveGlow**-Vocoder in ein 22,05-kHz-Audiosignal umgewandelt wird.
- **Strenge Eingabe- und Decoder-Beschränkungen:** Typografische Sonderzeichen (z. B. deutsche Anführungszeichen `„“`, Gedankenstriche `–`, `€`, `$`) müssen vorab per `clean_text()` bereinigt werden. Da der Decoder bei längeren Texten abbricht, muss der Input zusätzlich per `split_text()` in kurze Satz-Chunks (`<= 100` Zeichen) zerlegt werden.
- **Sprachlimitierung:** Das vortrainierte NVIDIA-Modell unterstützt ausschließlich Englisch; deutsche Texte werden mit starkem englischem Akzent vorgelesen.

### Referenz-Implementierung (Tacotron 2 + WaveGlow)
```python
import re
import numpy as np
import torch
from scipy.io.wavfile import write

def clean_text(text: str) -> str:
    replacements = {"„": '"', "“": '"', "‚": "'", "‘": "'", "—": "-", "–": "-", "€": "Euro", "$": "Dollar"}
    for k, v in replacements.items():
        text = text.replace(k, v)
    allowed = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789 .,!?\"'-äöüÄÖÜß"
    return "".join(c for c in text if c in allowed or c.isspace())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
utils = torch.hub.load("NVIDIA/DeepLearningExamples:torchhub", "nvidia_tts_utils", trust_repo=True)
tacotron2 = torch.hub.load("NVIDIA/DeepLearningExamples:torchhub", "nvidia_tacotron2", trust_repo=True).to(device).eval()
waveglow = torch.hub.load("NVIDIA/DeepLearningExamples:torchhub", "nvidia_waveglow", trust_repo=True)
waveglow = waveglow.remove_weightnorm(waveglow).to(device).eval()

sequences, lengths = utils.prepare_input_sequence([clean_text(short_sentence)])
with torch.no_grad():
    mel, _, _ = tacotron2.infer(sequences.to(device), lengths.to(device))
    audio = waveglow.infer(mel)[0].cpu().numpy()

write("audiofiles/english.wav", 22050, (audio / np.max(np.abs(audio))).astype(np.float32))
```

### Evaluierung & Bewertung

- **Englisch (0.0 / 5 Sterne Gesamtbewertung; Teilkriterien 1–3 / 5):**  
  Einzelne Wörter und Abkürzungen (`Dr.`) werden verständlich artikuliert, jedoch klingt die Ausgabe sehr monoton und durch die manuelle Chunk-Zerlegung ohne natürliche Satzpausen. Datumsformate werden nicht sauber erkannt, und Emotions- oder Temporegler fehlen.
- **Deutsch (0.0 / 5 Sterne):**  
  Für deutsche Texte unbrauchbar, da das englisch vortrainierte Modell deutsche Wörter mit englischer Phonetik liest, Zahlen auf Englisch ausspricht und deutsche Datumsangaben nicht erkennt.

---
## 4. Quantitativer Gesamtvergleich & Visualisierung

In [ ]:
benchmark_data = [
    {
        "Modell": "Edge-TTS",
        "Score EN (0-5)": 5.0,
        "Score DE (0-5)": 4.0,
        "Ø Kriterien EN": 4.86,
        "Ø Kriterien DE": 4.57,
        "Zahlen & Abkürzungen": "Sehr präzise (DE & EN)",
        "Betriebsart": "Cloud API (Microsoft Neural Voices)",
        "Status": "Produktiv übernommen (src/tts_edge.py)",
    },
    {
        "Modell": "PiperTTS",
        "Score EN (0-5)": 4.0,
        "Score DE (0-5)": 3.0,
        "Ø Kriterien EN": 4.57,
        "Ø Kriterien DE": 3.14,
        "Zahlen & Abkürzungen": "EN sehr gut; DE Schwächen bei 'Dr.' & Datum",
        "Betriebsart": "Lokal / Embedded (ONNX)",
        "Status": "Geeigneter Offline-Fallback für Englisch",
    },
    {
        "Modell": "Tacotron 2",
        "Score EN (0-5)": 0.0,
        "Score DE (0-5)": 0.0,
        "Ø Kriterien EN": 2.00,
        "Ø Kriterien DE": 1.14,
        "Zahlen & Abkürzungen": "Kein DE-Modell; Chunking & Text-Cleaning nötig",
        "Betriebsart": "Lokal (PyTorch / GPU)",
        "Status": "Verworfen (kein Deutsch, Decoder-Limit)",
    },
]

df_tts = pd.DataFrame(benchmark_data)
display(df_tts)

# Detail-Matrix über alle 7 Einzelkriterien (Skala 0-5)
criteria_details = pd.DataFrame(
    [
        {"Modell": "Edge-TTS (EN)", "Deutlichkeit": 5, "Natürlichkeit": 5, "Flüssigkeit": 5, "Interpunktion": 5, "Abkürzungen & Symbole": 5, "Zahlen & Datum": 5, "Anpassbarkeit": 4},
        {"Modell": "Edge-TTS (DE)", "Deutlichkeit": 5, "Natürlichkeit": 4, "Flüssigkeit": 5, "Interpunktion": 5, "Abkürzungen & Symbole": 5, "Zahlen & Datum": 4, "Anpassbarkeit": 4},
        {"Modell": "PiperTTS (EN)", "Deutlichkeit": 5, "Natürlichkeit": 3, "Flüssigkeit": 5, "Interpunktion": 5, "Abkürzungen & Symbole": 5, "Zahlen & Datum": 5, "Anpassbarkeit": 4},
        {"Modell": "PiperTTS (DE)", "Deutlichkeit": 3, "Natürlichkeit": 2, "Flüssigkeit": 4, "Interpunktion": 4, "Abkürzungen & Symbole": 2, "Zahlen & Datum": 3, "Anpassbarkeit": 4},
        {"Modell": "Tacotron 2 (EN)", "Deutlichkeit": 3, "Natürlichkeit": 2, "Flüssigkeit": 2, "Interpunktion": 1, "Abkürzungen & Symbole": 3, "Zahlen & Datum": 2, "Anpassbarkeit": 1},
        {"Modell": "Tacotron 2 (DE)", "Deutlichkeit": 0, "Natürlichkeit": 1, "Flüssigkeit": 1, "Interpunktion": 1, "Abkürzungen & Symbole": 3, "Zahlen & Datum": 1, "Anpassbarkeit": 1},
    ]
)
display(criteria_details)

# Balkendiagramm der Gesamtbewertungen (EN vs. DE)
fig, ax = plt.subplots(figsize=(8.5, 4))
x = range(len(df_tts))
ax.bar([i - 0.18 for i in x], df_tts["Score EN (0-5)"], width=0.35, label="Englisch (EN)", color="#2563eb")
ax.bar([i + 0.18 for i in x], df_tts["Score DE (0-5)"], width=0.35, label="Deutsch (DE)", color="#10b981")

ax.set_xticks(list(x))
ax.set_xticklabels(df_tts["Modell"])
ax.set_ylim(0, 5.5)
ax.set_ylabel("Gesamtbewertung (0–5 Sterne)")
ax.set_title("TTS-Modellvergleich: Englisch vs. Deutsch (Sprachqualität & linguistische Intelligenz)")
ax.legend()
plt.tight_layout()
plt.show()

### Optionaler Live-Test mit der produktiven Edge-TTS-Komponente

Die folgende Zelle synthetisiert die in `../data/reference_texts.json` hinterlegten Referenztexte mit **Edge-TTS** in den temporären Unterordner `../data/sample_audio/generated/` und spielt die generierten Audiodateien direkt im Notebook ab. *(Die eingecheckten Standard-Samples `sample_de.wav` und `sample_en.wav` werden über `python scripts/generate_sample_audio.py` erzeugt.)*

In [ ]:
import edge_tts
import nest_asyncio
from IPython.display import Audio, display

nest_asyncio.apply()

out_dir = Path("../data/sample_audio/generated")
out_dir.mkdir(parents=True, exist_ok=True)

voices = {
    "de": ("de-DE-KatjaNeural", out_dir / "tts_edge_de.mp3"),
    "en": ("en-US-AriaNeural", out_dir / "tts_edge_en.mp3"),
}

for lang_key, (voice_name, target_path) in voices.items():
    sample_text = ref_data["texts"].get(lang_key)
    if not sample_text:
        continue
    communicate = edge_tts.Communicate(sample_text, voice=voice_name)
    await communicate.save(str(target_path))
    print(f"Erstellt: {target_path} ({voice_name})")
    display(Audio(filename=str(target_path)))

---
## 5. Architekturentscheidung für die Pipeline

**Edge-TTS** wurde als produktives TTS-Backend gewählt:
1. **Sprachqualität & linguistische Intelligenz:** Edge-TTS erzielt sowohl im Englischen (**5.0 / 5**) als auch im Deutschen (**4.0 / 5**) die beste Bewertung und verarbeitet komplexe Eingaben wie Datumsangaben (`12. März 2024`), Abkürzungen (`Dr.`, `z. B.`), Prozentwerte (`92,4 %`) und Währungen ohne manuelle Textbereinigung oder Chunking.
2. **Abgrenzung zu den Alternativen:** **PiperTTS** liefert im Englischen ebenfalls starke Ergebnisse (**4.0 / 5**) und eignet sich als rein lokaler Offline-Fallback, zeigt im Deutschen (**3.0 / 5**) jedoch deutliche Aussprachefehler bei Abkürzungen und Daten. **Tacotron 2** scheidet aufgrund fehlender deutscher Sprachunterstützung und strikter Decoder-Längenbegrenzungen für den produktiven Einsatz aus.